# 04 / Giving an Agent Tools

**Deep Learning Indaba 2026 / Introduction to Agentic AI**

A **tool** is a function the agent can call to do something the model cannot do on its own.
The most striking example is reaching the **live world**: a model is frozen at training time,
so it cannot know what is happening *right now*. We give the agent a tool that calls a real
weather service, and suddenly it can tell you the current temperature in any city.

The model itself stays local and needs no key. The tool is the one place the agent reaches an
external service, which is exactly what tools are for.

> This notebook needs an internet connection for the weather call (Colab has one). The tool
> has a safe fallback, so a dropped connection will not crash the demo.

## 4.1 / Setup

In [ ]:
%pip install -q transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

def chat(messages, max_new_tokens=40, temperature=0.7):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature and temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("Ready on", model.device)

## 4.2 / The problem: the model cannot know what is happening now

Ask the model for the current weather in a city. It will answer confidently, but it has no
way to know: it was trained in the past and cannot see the live world. Whatever number it
gives is a guess.

In [ ]:
print(chat([{"role": "user", "content": "What is the current temperature in Nairobi right now? Answer in one short line."}],
           max_new_tokens=40, temperature=0.0))

## 4.3 / The tool: a real weather API (no key)

Here is the tool. It calls the free, no-key **Open-Meteo** service in two steps: look up the
city's coordinates, then fetch the current weather there. Everything is wrapped in a safe
fallback, so if the network is down the agent still behaves gracefully.

In [ ]:
import urllib.request, urllib.parse, json

def get_weather(city: str) -> str:
    """Tool: fetch the current weather for a city from the free Open-Meteo API (no key)."""
    try:
        geo_url = "https://geocoding-api.open-meteo.com/v1/search?" + urllib.parse.urlencode({"name": city, "count": 1})
        geo = json.load(urllib.request.urlopen(geo_url, timeout=10))
        if not geo.get("results"):
            return f"(could not find the city '{city}')"
        place = geo["results"][0]
        wx_url = "https://api.open-meteo.com/v1/forecast?" + urllib.parse.urlencode({
            "latitude": place["latitude"], "longitude": place["longitude"],
            "current": "temperature_2m,wind_speed_10m"})
        cur = json.load(urllib.request.urlopen(wx_url, timeout=10))["current"]
        return f"{place['name']}: {cur['temperature_2m']} degrees C, wind {cur['wind_speed_10m']} km/h"
    except Exception as e:
        return f"(weather service unavailable: {e})"

print(get_weather("Nairobi"))
print(get_weather("Lagos"))

## 4.4 / The agent uses the tool

The agent does three simple steps, and you can read each one in the code:

1. **Reason:** live weather is a job for the tool, not the model.
2. **Act:** call the weather API to get the real, current reading.
3. **Respond:** lead the reply with that real reading, and let the model add a friendly comment.

We place the tool's result ourselves, so the true number always shows up, even with a tiny
model that would otherwise ignore it.

In [ ]:
def agent(city):
    # 1) REASON: current weather is a job for the tool
    # 2) ACT: call the weather API
    fact = get_weather(city)
    print(f"[tool: get_weather('{city}') = {fact}]")
    # 3) RESPOND: lead with the real reading, then let the model add a warm sign-off.
    #    We keep the model's job easy (a cheerful line, no weather logic, no numbers),
    #    so a tiny model stays coherent and cannot mangle the real data.
    signoff = chat([
        {"role": "system", "content": "Reply with one short, cheerful sentence wishing the person a good day. Do not mention any numbers or weather details."},
        {"role": "user", "content": f"Someone asked about the weather in {city}."},
    ], max_new_tokens=25)
    return f"{fact}. {signoff}"

print(agent("Nairobi"))
print()
print(agent("Accra"))

Look at the difference from Section 4.2: the temperature is now **real and current**,
fetched live from the API, because the agent called the tool instead of guessing. The model
provides the friendly wording; the tool provides the fact.

> In a larger agent, the model itself decides *which* tool to call (this is called "function
> calling"). We kept that explicit here so the three steps stay obvious. The mechanism is the
> same.

## 4.5 / Why this matters

A tool is just a function, and swapping `get_weather` for something else gives the agent a
whole new ability:

- a **web search or news API**, so it can fetch current information it was never trained on,
- a **calculator**, so it can do exact arithmetic,
- a **database or an internal API**, so it can look up a record or take a real action,
- a **code runner**, so it can compute or analyse data on the spot.

The model supplies the reasoning; the tools supply the capabilities. This is the real bridge
from a chatbot that only talks to an agent that can actually *do* things. Notice the shape of
sovereignty here too: the model runs locally and privately, and *you* decide which external
services the agent may call, and you can point it at your own. You met a first tool back in
notebook `00` (the griot's sayings); combine tools with the memory from notebooks `01` and
`02`, and you have the core of a genuinely capable agent.